In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("../../data/raw/kathmandu_full_raw_2023_2024.csv")

# Datetime format conversion and sorting (although already sorted)
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

In [2]:
# Create target column. For instance i, target is PM2.5 of instance (i+24) i.e. PM2.5 after 24 hour.
data["target"] = data["pm2_5"].shift(-72)

In [3]:
# PM2.5 lag features, (t-1) to (t-12). Standard 12h short-range lag window; the recurring
# daily cycle is separately captured via hour_sin/hour_cos rather than extending lag depth.
for i in range(1, 13):
    data[f"pm2_5_lag_{i}"] = data["pm2_5"].shift(i)

In [4]:
# Pollutant lag depths, set individually based on cross-correlation analysis:
# - carbon_monoxide, nitrogen_dioxide, sulphur_dioxide: correlation decreases steadily
#   with lag, so only the last 3 hours carry meaningful signal.
# - ozone: correlation is near-zero at lag 0-1 but rises slowly, peaking around lag 10
#   (corr ~0.35), so a longer 12h window is kept to capture this delayed relationship.
# - pm10: kept at the original 3h depth (unchanged, not re-analyzed this round).
pollutant_lag_depths = {
    "pm10": 3,
    "carbon_monoxide": 3,
    "nitrogen_dioxide": 3,
    "sulphur_dioxide": 3,
    "ozone": 12,
}

for col, depth in pollutant_lag_depths.items():
    for i in range(1, depth + 1):
        data[f"{col}_lag_{i}"] = data[col].shift(i)

In [5]:
# Meteorological lag depths, set individually based on cross-correlation analysis:
# - temperature_2m: correlation is strongest in the last 3 hours before weakening, so
#   only lag 1-3 kept.
# - relative_humidity_2m: correlation is near-zero at lag 0-1 but peaks around lag 10,
#   staying strong through lag 12, so a 12h window is kept.
# - wind_speed_10m: correlation turns positive and peaks around lag 8, so a 12h window
#   is kept to cover the full rise.
# - surface_pressure: correlation increases and peaks around lag 11, so a 12h window
#   is kept.
meteo_lag_depths = {
    "temperature_2m": 3,
    "relative_humidity_2m": 12,
    "wind_speed_10m": 12,
    "surface_pressure": 12,
}

for col, depth in meteo_lag_depths.items():
    for i in range(1, depth + 1):
        data[f"{col}_lag_{i}"] = data[col].shift(i)

# Wind direction (raw degrees) is shifted here too, but only as an intermediate —
# it gets converted to lagged cosine features in CyclicalFeatures below, and the
# raw shifted columns are dropped there (not usable directly since 0-360 wraps around).
for i in range(1, 13):
    data[f"wind_direction_10m_lag_{i}"] = data["wind_direction_10m"].shift(i)

In [6]:
# Handling null values. First 12 rows dropped due to the 12h lag window used across
# ozone, humidity, wind speed, surface pressure, and wind direction.
data = data.dropna().reset_index(drop=True)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17460 entries, 0 to 17459
Data columns (total 100 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   time                         17460 non-null  datetime64[us]
 1   pm2_5                        17460 non-null  float64       
 2   pm10                         17460 non-null  float64       
 3   carbon_monoxide              17460 non-null  int64         
 4   nitrogen_dioxide             17460 non-null  float64       
 5   sulphur_dioxide              17460 non-null  float64       
 6   ozone                        17460 non-null  int64         
 7   temperature_2m               17460 non-null  float64       
 8   relative_humidity_2m         17460 non-null  int64         
 9   wind_speed_10m               17460 non-null  float64       
 10  wind_direction_10m           17460 non-null  int64         
 11  surface_pressure             17460 non-null  float6

In [7]:
# All numeric columns for scaling (linear models only). Raw wind_direction_10m and its
# lagged raw versions are excluded here since they get replaced by cosine-transformed
# features in CyclicalFeatures, not used directly (0-360 degree wraparound isn't
# meaningful to a linear model without transformation).
wind_dir_raw_cols = ["wind_direction_10m"] + [f"wind_direction_10m_lag_{i}" for i in range(1, 13)]

numerical_cols = list(
    data.drop(columns=["time", "target"] + wind_dir_raw_cols).columns
)

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin

class MeterologicalBinFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def set_output(self, transform=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["temp_bin"] = pd.cut(
            X["temperature_2m"],
            bins=[-np.inf, 10, 20, np.inf],
            labels=["low", "medium", "high"]
        )

        X["humidity_bin"] = pd.cut(
            X["relative_humidity_2m"],
            bins=[-np.inf, 40, np.inf],
            labels=["low", "normal"]
        )

        X["wind_speed_bin"] = pd.cut(
            X["wind_speed_10m"],
            bins=[-np.inf, 3, 6, 12, 15, np.inf],
            labels=["calm", "light", "moderate", "strong", "very_strong"]
        )

        X["wind_direction_bin"] = np.where(
            (X["wind_direction_10m"] > 120) & (X["wind_direction_10m"] <= 240),
            "mid",
            "outer"
        )

        X["surface_pressure_bin"] = pd.cut(
            X["surface_pressure"],
            bins=[-np.inf, 865, 870, 875, np.inf],
            labels=["very_low", "low", "normal", "high"]
        )

        return X

In [9]:
class TemporalBinFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def set_output(self, transform=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = X["time"].dt.hour
        X["month"] = X["time"].dt.month

        # Hour-of-day bins, built from actual point-11 hourly PM2.5 averages, not fixed
        # morning/afternoon labels. Four contiguous circular segments:
        #   peak (20,21,22,23,0): 36-40 ug/m3, the nightly high
        #   morning_decline (1-8): 29-36 ug/m3, declining from the peak toward the trough
        #   trough (9-16): 23-26 ug/m3, the exceptionally low midday period
        #   evening_rise (17-19): 29-36 ug/m3, rising back up toward the peak
        X["hour_bin"] = np.select(
            [
                X["hour"].isin([20, 21, 22, 23, 0]),
                X["hour"].isin([1, 2, 3, 4, 5, 6, 7, 8]),
                X["hour"].isin([9, 10, 11, 12, 13, 14, 15, 16]),
                X["hour"].isin([17, 18, 19]),
            ],
            ["peak", "morning_decline", "trough", "evening_rise"],
            default="unknown"
        )

        # Month bins, built from actual point-13 monthly PM2.5 averages, kept as
        # contiguous sequential groups (no non-adjacent months combined):
        #   winter_high (Nov-Feb): 38.6-48.3 ug/m3
        #   spring_moderate (Mar-Jun): 27.4-31.7 ug/m3
        #   monsoon_low (Jul-Aug): 15.0-17.6 ug/m3, the cleanest period
        #   post_monsoon_transition (Sep-Oct): 17.7-24.5 ug/m3, rising back up
        X["month_bin"] = np.select(
            [
                X["month"].isin([11, 12, 1, 2]),
                X["month"].isin([3, 4, 5, 6]),
                X["month"].isin([7, 8]),
                X["month"].isin([9, 10]),
            ],
            ["winter_high", "spring_moderate", "monsoon_low", "post_monsoon_transition"],
            default="unknown"
        )

        X = X.drop(columns=["hour", "month"])

        return X

categorical_cols = [
    "temp_bin",
    "humidity_bin",
    "wind_speed_bin",
    "wind_direction_bin",
    "surface_pressure_bin",
    "hour_bin",
    "month_bin",
]

In [10]:
class CyclicalFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def set_output(self, transform=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = X["time"].dt.hour
        X["month"] = X["time"].dt.month
        X["day_of_week"] = X["time"].dt.dayofweek

        X["hour_sin"] = np.sin(2*np.pi*X["hour"]/24)
        X["hour_cos"] = np.cos(2*np.pi*X["hour"]/24)

        # Day-of-week cyclical encoding, kept. Point 12 showed only a small average
        # PM2.5 difference across weekdays (~29.7 to ~31.9) — too weak to justify a
        # categorical bin, but kept here as a low-cost continuous cyclical feature.
        X["dow_sin"] = np.sin(2*np.pi*X["day_of_week"]/7)
        X["dow_cos"] = np.cos(2*np.pi*X["day_of_week"]/7)

        X["month_sin"] = np.sin(2*np.pi*X["month"]/12)
        X["month_cos"] = np.cos(2*np.pi*X["month"]/12)

        # Wind direction: current-hour sin/cos kept as before.
        X["wind_dir_sin"] = np.sin(np.deg2rad(X["wind_direction_10m"]))
        X["wind_dir_cos"] = np.cos(np.deg2rad(X["wind_direction_10m"]))

        # Wind direction cosine, lagged 1-12h. Cross-correlation showed wind_dir_cos's
        # relationship with PM2.5 shifts from about -0.22 at lag 0 to a peak of about
        # +0.20 around lag 9-10, a meaningfully different relationship than the current
        # value alone — sin was not extended since cos showed the stronger relationship.
        for i in range(1, 13):
            X[f"wind_dir_cos_lag_{i}"] = np.cos(np.deg2rad(X[f"wind_direction_10m_lag_{i}"]))

        drop_cols = ["hour", "month", "day_of_week", "time", "wind_direction_10m"]
        drop_cols += [f"wind_direction_10m_lag_{i}" for i in range(1, 13)]
        X = X.drop(columns=drop_cols)

        return X

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

pipeline_linear = Pipeline([
    ("meterological", MeterologicalBinFeatures()),
    ("temporal", TemporalBinFeatures()),
    ("cyclical", CyclicalFeatures()),
    ("column_transform", ColumnTransformer(
        transformers=[
            ("onehotencoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
            ("scaler", StandardScaler(), numerical_cols),
        ],
        remainder="passthrough"
    )),
]).set_output(transform="pandas")

X = data.drop(columns=["target"])
y = data["target"]

split = int(np.ceil(0.8 * len(X)))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

X_train_t = pipeline_linear.fit_transform(X_train)
X_test_t = pipeline_linear.transform(X_test)

linear_reg = LinearRegression()
linear_reg.fit(X_train_t, y_train)

y_pred = linear_reg.predict(X_test_t)
print(root_mean_squared_error(y_pred, y_test))
print(mean_absolute_error(y_pred, y_test))

26.92951854388484
16.736012798024245


In [12]:
X_train_t.to_csv("../../data/processed/72_hour_prediction/X_train_linear.csv", index=False)
X_test_t.to_csv("../../data/processed/72_hour_prediction/X_test_linear.csv", index=False)
y_train.to_csv("../../data/processed/72_hour_prediction/y_train.csv", index=False)
y_test.to_csv("../../data/processed/72_hour_prediction/y_test.csv", index=False)

In [13]:
pipeline_tree = Pipeline([
    ("cyclical", CyclicalFeatures()), # add cyclical features, also remove time and wind direction column
])

In [14]:
X = data.drop(columns=["target"])
y = data["target"]

split = int(np.ceil(0.8 * len(X)))

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [15]:
X_train_t = pipeline_tree.fit_transform(X_train)
X_test_t = pipeline_tree.transform(X_test)

In [16]:
X_train_t.to_csv("../../data/processed/72_hour_prediction/X_train_tree.csv", index=False)
X_test_t.to_csv("../../data/processed/72_hour_prediction/X_test_tree.csv", index=False)